# Notebook 02: Numerical and Categorical Features

## Why This Matters

Almost every real-world dataset mixes numerical and categorical columns, and handling them incorrectly is one of the most common sources of silent bugs in ML systems. Encoding a nominal variable as raw integers (e.g., red=1, green=2, blue=3) implies an ordinal relationship and distance structure that does not exist, which corrupts both linear and distance-based models. Conversely, one-hot encoding a high-cardinality categorical (e.g., ZIP code with 40,000 levels) explodes the feature space and makes training impractical. Knowing which encoding to choose — and when to apply target encoding, embeddings, or frequency encoding — is consistently tested in ML interviews because it separates people who understand model internals from people who just call `.fit()`.

## 1. Numerical Feature Types and Pipelines

Numerical features fall into two conceptual categories:
- **Continuous**: can take any value in a range (age, temperature, price). Handle with scaling and outlier treatment.
- **Discrete / count**: integer-valued (number of items, event counts). Often follow Poisson or negative-binomial distributions; log1p or sqrt transforms help.

For missing values in numerical features the standard options are:
1. **Mean imputation** — fast, degrades variance estimates, assumes MAR (Missing At Random)
2. **Median imputation** — robust to outliers, still assumes MAR
3. **KNN imputation** — uses similar rows; expensive at scale
4. **Indicator + imputation** — add a binary `was_missing` flag before imputing; lets the model learn that missingness itself is informative

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Build a synthetic dataset with mixed types and missingness
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age':       np.random.randint(18, 70, n).astype(float),
    'income':    np.random.exponential(50000, n),
    'n_purchases': np.random.poisson(5, n).astype(float),
    'city':      np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix'], n),
    'plan':      np.random.choice(['free', 'basic', 'premium'], n, p=[0.5, 0.3, 0.2]),
    'device':    np.random.choice(['mobile', 'desktop', 'tablet'], n, p=[0.6, 0.3, 0.1]),
})
df['churn'] = ((df['income'] < 30000) | (df['n_purchases'] < 3)).astype(int)

# Introduce ~15% missingness in numerical columns
for col in ['age', 'income', 'n_purchases']:
    mask = np.random.rand(n) < 0.15
    df.loc[mask, col] = np.nan

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
df.head()

In [ ]:
# Compare imputation strategies
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder

num_cols = ['age', 'income', 'n_purchases']
cat_cols = ['city', 'plan', 'device']
y = df['churn']

strategies = ['mean', 'median', 'most_frequent']
for strategy in strategies:
    num_pipe = Pipeline([
        ('impute', SimpleImputer(strategy=strategy)),
        ('scale',  StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    ct = ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols),
    ])
    pipe = Pipeline([('ct', ct), ('lr', LogisticRegression(max_iter=500))])
    scores = cross_val_score(pipe, df[num_cols + cat_cols], y, cv=5, scoring='roc_auc')
    print(f"Imputation='{strategy:12s}' | AUC = {scores.mean():.4f} ± {scores.std():.4f}")

## 2. Binning Continuous Features

**Binning** (also called discretization) converts a continuous feature into ordered categorical buckets. Use cases:
- When the relationship between feature and target is non-monotonic (e.g., risk is high for very young *and* very old borrowers)
- When you want to add a linear-model-friendly non-linearity without polynomial expansion
- When the raw value is noisy but the bucket is stable (e.g., age groups instead of exact age)

Two strategies in sklearn:
- **`KBinsDiscretizer(strategy='uniform')`** — equal-width bins; susceptible to outliers
- **`KBinsDiscretizer(strategy='quantile')`** — equal-frequency bins; handles skew better

Caution: binning always loses information within each bin — only use when the signal you want to capture is at the *between-bin* level.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# Fill missing income for this demo
income = df['income'].fillna(df['income'].median()).values.reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(income, bins=40, edgecolor='k', alpha=0.7)
axes[0].set_title('Original Income (continuous)', fontsize=10)

for ax, strategy in zip(axes[1:], ['uniform', 'quantile']):
    kbd = KBinsDiscretizer(n_bins=8, encode='ordinal', strategy=strategy)
    binned = kbd.fit_transform(income).ravel()
    ax.hist(binned, bins=8, edgecolor='k', alpha=0.7)
    ax.set_title(f'Binned ({strategy})', fontsize=10)
    ax.set_xlabel('Bin index')
    print(f"{strategy:10s} bin edges: {kbd.bin_edges_[0].round(0)}")

plt.suptitle('Continuous → Binned Feature', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Categorical Encoding: The Decision Tree

The choice of encoding depends on three things: **cardinality** (how many unique values?), **model type** (tree vs. linear), and **whether the category has an order**.

```
Categorical feature
├── Ordinal (low < medium < high)  →  OrdinalEncoder with explicit ordering
└── Nominal (no order)
    ├── Low cardinality (< ~15 levels)
    │   ├── Linear / SVM model  →  OneHotEncoder
    │   └── Tree model          →  OrdinalEncoder (trees handle integers fine)
    └── High cardinality (> ~15 levels)
        ├── Sufficient data     →  TargetEncoder (with cross-fitting to avoid leakage)
        ├── Cold-start concern  →  FrequencyEncoder / CountEncoder
        └── Embeddings needed   →  Learned embeddings (deep learning)
```

## 4. One-Hot Encoding

One-hot encoding (OHE) creates one binary column per category level. It is the correct default for nominal variables in linear models because it makes no assumption about ordering or distance between levels.

Key pitfall: **the dummy variable trap**. With k levels, OHE produces k columns, but only k-1 are linearly independent. For linear models with an intercept, always drop one category (`drop='first'`). Tree models are not affected.

Scalability concern: with a feature that has 10,000 unique cities, OHE produces 10,000 columns. Memory and compute explode. Use sparse matrices (`sparse_output=True` in sklearn) or switch to a compact encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# OHE on the plan column (ordinal but let's see OHE first)
ohe = OneHotEncoder(sparse_output=False, drop='first')
plan_ohe = ohe.fit_transform(df[['plan']])
print("OHE categories:", ohe.categories_)
print("OHE output shape:", plan_ohe.shape)
print("OHE column names:", ohe.get_feature_names_out())
print()

# Ordinal encoding with explicit order for plan
oe = OrdinalEncoder(categories=[['free', 'basic', 'premium']])
plan_ord = oe.fit_transform(df[['plan']])
print("Ordinal encoding (plan):")
sample = pd.DataFrame({'plan': df['plan'].head(8).values,
                        'encoded': plan_ord[:8].ravel()})
print(sample.drop_duplicates().sort_values('encoded'))

## 5. Target Encoding (Mean Encoding)

**Target encoding** replaces each category level with the mean of the target variable for that level. It reduces cardinality to a single numeric column and naturally captures the relationship between the category and the target.

The major risk is **target leakage**: if you compute the mean over the entire training set and then use it as a feature, the model sees label information directly. The fix is **cross-fitting**: encode each fold using the other folds, exactly as cross-validation works. sklearn's `TargetEncoder` does this automatically.

A secondary problem is **rare category noise**: a category with only 2 observations will have a noisy mean. **Smoothing** addresses this by blending the category mean toward the global mean:

$$\hat{\mu}_c = \frac{n_c \cdot \bar{y}_c + \alpha \cdot \bar{y}_{global}}{n_c + \alpha}$$

where α is the smoothing strength (higher α → more regularization toward global mean).

In [ ]:
from sklearn.preprocessing import TargetEncoder

# Simulate a high-cardinality categorical: user_id (many unique values)
np.random.seed(0)
n = 1000
n_cities = 50
cities = [f'city_{i:03d}' for i in range(n_cities)]
# Cities have true varying churn rates
city_churn_rate = {c: np.clip(np.random.beta(2, 5), 0.05, 0.95) for c in cities}
city_col = np.random.choice(cities, n)
y_hc = np.array([np.random.binomial(1, city_churn_rate[c]) for c in city_col])
X_hc = pd.DataFrame({'city': city_col, 'feature2': np.random.randn(n)})

# sklearn TargetEncoder uses cross-fitting automatically
te = TargetEncoder(smooth='auto', target_type='binary')
X_encoded = te.fit_transform(X_hc[['city']], y_hc)
print("TargetEncoder output shape:", X_encoded.shape)
print("Sample encoded values (first 10 unique cities):")
sample_df = pd.DataFrame({'city': city_col, 'encoded': X_encoded.ravel()})
print(sample_df.groupby('city')['encoded'].first().head(10).round(4))

In [ ]:
# Implement smoothed target encoding from scratch to show the math
def smooth_target_encode(series, target, alpha=10):
    """Replace categories with smoothed mean of target.
    alpha: smoothing strength. Higher = stronger regularization toward global mean.
    """
    global_mean = target.mean()
    stats = target.groupby(series).agg(['mean', 'count'])
    stats['smoothed'] = (stats['count'] * stats['mean'] + alpha * global_mean) / (stats['count'] + alpha)
    return series.map(stats['smoothed'])

plan_series = df['plan']
encoded = smooth_target_encode(plan_series, df['churn'], alpha=5)
print("Smoothed target encoding for 'plan' feature:")
print(pd.DataFrame({'plan': plan_series, 'encoded': encoded}).drop_duplicates().sort_values('encoded'))

## 6. Frequency and Hash Encoding

**Frequency encoding** replaces each category with its count or frequency in the training set. It is fast, compact, and preserves relative rarity — useful when how common a value is correlates with the target (e.g., rare device types churn more).

**Hash encoding** (feature hashing / the hashing trick) maps arbitrary strings to a fixed-size vector using a hash function. It handles unseen categories at serving time naturally and has constant memory cost regardless of cardinality — at the cost of potential hash collisions. Used extensively in large-scale production systems (Vowpal Wabbit, Spark ML).

In [ ]:
from sklearn.feature_extraction import FeatureHasher

# Frequency encoding
city_freq = df['city'].value_counts(normalize=True)
df['city_freq'] = df['city'].map(city_freq)
print("Frequency encoding for city:")
print(df.groupby('city')['city_freq'].first().sort_values(ascending=False))

print()
# Hash encoding using sklearn FeatureHasher
hasher = FeatureHasher(n_features=8, input_type='string')
# FeatureHasher expects an iterable of dicts or lists of strings per sample
city_hashed = hasher.transform([[c] for c in df['city'].head(10)])
print("\nHash encoding output shape (10 samples, 8 hash buckets):")
print(city_hashed.toarray())

## 7. Handling Rare and Unseen Categories

Two critical production concerns:

1. **Rare categories** (few training examples): Their statistics are noisy. Group them into an `'OTHER'` bucket or apply smoothed target encoding.
2. **Unseen categories** (appear at serving time but not in training): OHE will error or produce all-zeros; OrdinalEncoder will error by default. Always set `handle_unknown='ignore'` in sklearn OHE and add an `OTHER` category in ordinal encoders.

The pattern below shows a robust categorical pipeline.

In [ ]:
def collapse_rare(series, min_freq=0.02, other_label='OTHER'):
    """Replace low-frequency categories with OTHER."""
    freq = series.value_counts(normalize=True)
    rare = freq[freq < min_freq].index
    return series.replace(rare, other_label)

# Add some rare cities for the demo
df_demo = df.copy()
df_demo['city'] = df_demo['city'].copy()
df_demo.loc[:5, 'city'] = 'TinyTown'

print("Before collapsing rare:")
print(df_demo['city'].value_counts())
print()
df_demo['city_clean'] = collapse_rare(df_demo['city'], min_freq=0.03)
print("After collapsing rare (min_freq=3%):")
print(df_demo['city_clean'].value_counts())

## Summary

| Situation | Encoding | Why |
|-----------|----------|-----|
| Nominal, low cardinality, linear model | OneHotEncoder (drop='first') | Avoids dummy trap; no false ordinal signal |
| Ordinal (explicit ordering) | OrdinalEncoder with category order | Encodes true order without spurious distances |
| High cardinality, sufficient data | TargetEncoder (with cross-fit) | Single column, captures target relationship |
| High cardinality, production serving | FrequencyEncoder or HashEncoder | Handles unseen categories; no leakage risk |
| Rare categories | Collapse to OTHER first | Reduces noise and prevents model from fitting noise |
| Missing numerical values | Median + missingness indicator | Robust to outliers; preserves signal in missingness |
| Non-monotonic continuous feature | KBinsDiscretizer (quantile) | Captures non-linear threshold effects cleanly |